# Notebook 04 — StratLake Feature Data Series Index and Dual-Session Setup

This notebook starts the StratLake feature-data tutorial series while preserving the latest `fintech-market-ingestion` notebook session and archive conventions.

The series introduces **StratLake Trade Engine** as the feature-generation and research workspace that consumes curated market data from `fintech-market-ingestion`.

This notebook intentionally keeps two session IDs:

```text
FINTECH_SESSION_ID   -> upstream ingestion / curated-data workspace
STRATLAKE_SESSION_ID -> downstream feature/research workspace
```

Do not collapse these into a single identifier. The Fintech session owns curated market data and archive packs; the StratLake session owns feature-generation setup and downstream research artifacts.

Series scope:

```text
Notebook 04 — StratLake Feature Data Series Index and Dual-Session Setup
Notebook 05 — StratLake Q1 Feature Data Generation
Notebook 06 — StratLake Session Save and Restore
Notebook 07 — StratLake Session Archive and Restore
```

The main flow is:

```text
Fintech curated market data
        ↓
Fintech SESSION_ID-scoped Drive persistence / archive restore readiness
        ↓
StratLake feature generation for Q1
        ↓
StratLake session save / restore
        ↓
StratLake session archive / restore
```

## What this series demonstrates

This presentation series focuses on StratLake concepts while respecting the latest Fintech storage milestone pattern:

- initializing a Fintech notebook session as the upstream curated-data provider
- preserving `FINTECH_SESSION_ID` from the Fintech session manifest
- keeping Fintech Drive storage scoped by `{FINTECH_SESSION_ID}`
- preparing Fintech archive-pack IDs and restore paths for curated data
- initializing a StratLake notebook workspace
- preserving `STRATLAKE_SESSION_ID` from StratLake session metadata
- connecting StratLake to an explicit external curated-data root
- generating Q1 feature data from curated market data
- saving and restoring a StratLake notebook session
- creating and restoring a portable StratLake session archive

The Fintech workspace appears only as the upstream curated-data source, but its session/archive identity is kept explicit so later notebooks can restore the same data source deterministically.

## Install required presentation packages

Run this cell in a fresh notebook runtime.

This tutorial series uses `fintech-market-ingestion` as the source of curated market data and `stratlake-trade-engine` as the feature-generation and research platform.

In [ ]:
!pip install "pandas-market-calendars>=5.0"
!pip install -i https://test.pypi.org/simple/ fintech-market-ingestion
!pip install -i https://test.pypi.org/simple/ stratlake-trade-engine

## Verify required CLI commands

These notebooks use both `fintech-market-ingestion` and `stratlake-trade-engine` commands.

The latest Fintech milestone adds archive-oriented storage helpers, so this setup notebook checks for both the session initializer and archive backup CLI.

If any command is missing, rerun the install cell above.

In [ ]:
import shutil

required_commands = [
    "fintech-init-project",
    "fintech-save-session",
    "fintech-backup-data",
    "stratlake-init-session",
    "stratlake-build-features",
    "stratlake-session-export",
    "stratlake-session-import",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
]

for command in required_commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'NOT FOUND'}")

## Authorize Google Drive access

Run this cell in Google Colab to authorize Google Drive access.

The packages do not mount Google Drive for you. Drive access is user-initiated here, and later commands treat Drive as a mounted filesystem path.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Define shared Fintech and StratLake paths

The Fintech workspace provides curated market data.

The StratLake workspace consumes that curated data through an explicit `marketlake_root` and generates feature data for research workflows.

Google Drive is used as persistence and archive storage only. The active runtime remains local-first for faster Colab execution.

The Drive layout below is intentionally session-scoped:

```text
/content/drive/MyDrive/{DRIVE_FOLDER_NAME}/
  fintech-market-ingestion/sessions/{FINTECH_SESSION_ID}/
  stratlake-trade-engine/sessions/{STRATLAKE_SESSION_ID}/
```

Update `DRIVE_FOLDER_NAME` to match the folder name you use in your Google Drive before running Drive-dependent cells.

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

FINTECH_ROOT = Path("/content/fintech-market-ingestion-demo")
STRATLAKE_ROOT = Path("/content/stratlake-trade-engine-demo")

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"

FINTECH_SESSION_NAME = f"fintech_stratlake_input_{TIMESTAMP_UTC}"
STRATLAKE_SESSION_NAME = f"stratlake_q1_features_{TIMESTAMP_UTC}"

DRIVE_FOLDER_NAME_IS_PLACEHOLDER = DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME"
if DRIVE_FOLDER_NAME_IS_PLACEHOLDER:
    print("Update DRIVE_FOLDER_NAME before running Drive-dependent cells in Colab.")

print("FINTECH_ROOT:", FINTECH_ROOT)
print("STRATLAKE_ROOT:", STRATLAKE_ROOT)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("FINTECH_DRIVE_ROOT:", FINTECH_DRIVE_ROOT)
print("STRATLAKE_DRIVE_ROOT:", STRATLAKE_DRIVE_ROOT)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("FINTECH_SESSION_NAME:", FINTECH_SESSION_NAME)
print("STRATLAKE_SESSION_NAME:", STRATLAKE_SESSION_NAME)

## Initialize the Fintech project session

This session provides the curated input data location for StratLake.

This tutorial series does not go deep into data ingestion. It assumes curated data exists from the earlier Fintech notebooks or can be restored into the Fintech local workspace from a `{FINTECH_SESSION_ID}`-scoped archive pack.

The important rule is that downstream notebooks should discover `FINTECH_SESSION_ID` from the manifest rather than hardcoding a session folder.

In [ ]:
!fintech-init-project \
  --root {FINTECH_ROOT.as_posix()} \
  --notebooks \
  --with-session \
  --session-name {FINTECH_SESSION_NAME}

## Extract `FINTECH_SESSION_ID`

The Fintech session ID is read from the latest Fintech project-session manifest.

This is the canonical notebook variable for Fintech Drive paths and archive IDs. Preserve it for later notebooks.

In [ ]:
fintech_manifest_paths = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime,
)

if not fintech_manifest_paths:
    raise FileNotFoundError("No Fintech session_manifest.json files found.")

FINTECH_SESSION_MANIFEST_PATH = fintech_manifest_paths[-1]
FINTECH_SESSION_MANIFEST = json.loads(FINTECH_SESSION_MANIFEST_PATH.read_text(encoding="utf-8"))
FINTECH_SESSION_ID = FINTECH_SESSION_MANIFEST["session_id"]

print("FINTECH_SESSION_MANIFEST_PATH:", FINTECH_SESSION_MANIFEST_PATH)
print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)

## Initialize the StratLake project session

StratLake uses an explicit session-first workspace that records the selected project root, external MarketLake root, and Drive persistence root.

`marketlake_root` points to the Fintech curated-data directory, not to a copied StratLake-owned dataset. This keeps Fintech as the upstream data provider and StratLake as the feature/research consumer.

In [ ]:
!stratlake-init-session \
  --root {STRATLAKE_ROOT.as_posix()} \
  --project-name {STRATLAKE_SESSION_NAME} \
  --marketlake-root {MARKETLAKE_ROOT.as_posix()} \
  --drive-root {DRIVE_ROOT.as_posix()} \
  --enable-drive-persistence \
  --notebook-configs

## Extract `STRATLAKE_SESSION_ID`

StratLake session metadata lives under `.stratlake/session.json`.

Some StratLake session metadata uses `project_name` as the stable identifier. This cell extracts `session_id` when present and otherwise falls back to `project_name`.

In [ ]:
STRATLAKE_SESSION_FILE = STRATLAKE_ROOT / ".stratlake" / "session.json"

if not STRATLAKE_SESSION_FILE.exists():
    raise FileNotFoundError(f"Missing StratLake session file: {STRATLAKE_SESSION_FILE}")

STRATLAKE_SESSION_MANIFEST = json.loads(STRATLAKE_SESSION_FILE.read_text(encoding="utf-8"))
STRATLAKE_SESSION_ID = (
    STRATLAKE_SESSION_MANIFEST.get("session_id")
    or STRATLAKE_SESSION_MANIFEST.get("project_name")
    or STRATLAKE_SESSION_NAME
)

print("STRATLAKE_SESSION_FILE:", STRATLAKE_SESSION_FILE)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)

## Create SESSION_ID-based Google Drive folders and archive IDs

This cell creates Drive folders only if they do not already exist.

Both Fintech and StratLake get session-scoped persistence folders. Fintech also gets a session-scoped backup/archive root for curated-data archive-pack workflows.

Archive IDs are derived from the active session IDs:

```text
FINTECH_ARCHIVE_ID   = curated-data-{FINTECH_SESSION_ID}
STRATLAKE_ARCHIVE_ID = stratlake-session-{STRATLAKE_SESSION_ID}
```

These archive IDs are transfer/restore identifiers, not canonical data sources.

In [ ]:
if DRIVE_FOLDER_NAME_IS_PLACEHOLDER:
    raise RuntimeError("Update DRIVE_FOLDER_NAME before creating Drive directories.")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Google Drive in Colab before creating Drive directories.")

FINTECH_DRIVE_SESSIONS_ROOT = FINTECH_DRIVE_ROOT / "sessions"
STRATLAKE_DRIVE_SESSIONS_ROOT = STRATLAKE_DRIVE_ROOT / "sessions"

FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / FINTECH_SESSION_ID
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / STRATLAKE_SESSION_ID

FINTECH_DRIVE_BACKUP_ROOT = FINTECH_DRIVE_SESSION_ROOT / "backups"
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"

FINTECH_BACKUP_PACK_DIR = FINTECH_DRIVE_BACKUP_ROOT / FINTECH_ARCHIVE_ID
STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

for path in [
    FINTECH_DRIVE_SESSION_ROOT,
    STRATLAKE_DRIVE_SESSION_ROOT,
    FINTECH_DRIVE_BACKUP_ROOT,
    STRATLAKE_DRIVE_ARCHIVE_ROOT,
    FINTECH_BACKUP_PACK_DIR,
    STRATLAKE_ARCHIVE_PACK_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

FINTECH_ROOT_STR = FINTECH_ROOT.as_posix()
STRATLAKE_ROOT_STR = STRATLAKE_ROOT.as_posix()
MARKETLAKE_ROOT_STR = MARKETLAKE_ROOT.as_posix()
DRIVE_ROOT_STR = DRIVE_ROOT.as_posix()

FINTECH_DRIVE_ROOT_STR = FINTECH_DRIVE_ROOT.as_posix()
STRATLAKE_DRIVE_ROOT_STR = STRATLAKE_DRIVE_ROOT.as_posix()
FINTECH_DRIVE_SESSION_ROOT_STR = FINTECH_DRIVE_SESSION_ROOT.as_posix()
STRATLAKE_DRIVE_SESSION_ROOT_STR = STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
FINTECH_DRIVE_BACKUP_ROOT_STR = FINTECH_DRIVE_BACKUP_ROOT.as_posix()
STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()
FINTECH_BACKUP_PACK_DIR_STR = FINTECH_BACKUP_PACK_DIR.as_posix()
STRATLAKE_ARCHIVE_PACK_DIR_STR = STRATLAKE_ARCHIVE_PACK_DIR.as_posix()

print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("FINTECH_ARCHIVE_ID:", FINTECH_ARCHIVE_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)
print("FINTECH_DRIVE_SESSION_ROOT:", FINTECH_DRIVE_SESSION_ROOT)
print("STRATLAKE_DRIVE_SESSION_ROOT:", STRATLAKE_DRIVE_SESSION_ROOT)
print("FINTECH_DRIVE_BACKUP_ROOT:", FINTECH_DRIVE_BACKUP_ROOT)
print("STRATLAKE_DRIVE_ARCHIVE_ROOT:", STRATLAKE_DRIVE_ARCHIVE_ROOT)
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR)
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR)

## Optional: choose previous Drive sessions for restore

Leave these values set to the current session IDs unless you intentionally want to restore from an older Drive session.

For a previous Fintech session, update both:

```python
RESTORE_FINTECH_SESSION_ID = "..."
RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
```

The default keeps restore previews tied to the current `{FINTECH_SESSION_ID}`.

In [ ]:
available_fintech_sessions = sorted(
    path.name for path in FINTECH_DRIVE_SESSIONS_ROOT.glob("*") if path.is_dir()
)
available_stratlake_sessions = sorted(
    path.name for path in STRATLAKE_DRIVE_SESSIONS_ROOT.glob("*") if path.is_dir()
)

print("Available Fintech Drive sessions:")
for session_id in available_fintech_sessions:
    print(" -", session_id)

print("\nAvailable StratLake Drive sessions:")
for session_id in available_stratlake_sessions:
    print(" -", session_id)

RESTORE_FINTECH_SESSION_ID = FINTECH_SESSION_ID
RESTORE_STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID

RESTORE_FINTECH_ARCHIVE_ID = f"curated-data-{RESTORE_FINTECH_SESSION_ID}"
RESTORE_STRATLAKE_ARCHIVE_ID = f"stratlake-session-{RESTORE_STRATLAKE_SESSION_ID}"

RESTORE_FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_SESSIONS_ROOT / RESTORE_FINTECH_SESSION_ID
RESTORE_STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_SESSIONS_ROOT / RESTORE_STRATLAKE_SESSION_ID

RESTORE_FINTECH_DRIVE_BACKUP_ROOT = RESTORE_FINTECH_DRIVE_SESSION_ROOT / "backups"
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT = RESTORE_STRATLAKE_DRIVE_SESSION_ROOT / "archives"

RESTORE_FINTECH_BACKUP_PACK_DIR = RESTORE_FINTECH_DRIVE_BACKUP_ROOT / RESTORE_FINTECH_ARCHIVE_ID
RESTORE_STRATLAKE_ARCHIVE_PACK_DIR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT / RESTORE_STRATLAKE_ARCHIVE_ID

RESTORE_FINTECH_DRIVE_SESSION_ROOT_STR = RESTORE_FINTECH_DRIVE_SESSION_ROOT.as_posix()
RESTORE_STRATLAKE_DRIVE_SESSION_ROOT_STR = RESTORE_STRATLAKE_DRIVE_SESSION_ROOT.as_posix()
RESTORE_FINTECH_DRIVE_BACKUP_ROOT_STR = RESTORE_FINTECH_DRIVE_BACKUP_ROOT.as_posix()
RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT_STR = RESTORE_STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix()
RESTORE_FINTECH_BACKUP_PACK_DIR_STR = RESTORE_FINTECH_BACKUP_PACK_DIR.as_posix()
RESTORE_STRATLAKE_ARCHIVE_PACK_DIR_STR = RESTORE_STRATLAKE_ARCHIVE_PACK_DIR.as_posix()

print("\nRESTORE_FINTECH_SESSION_ID:", RESTORE_FINTECH_SESSION_ID)
print("RESTORE_FINTECH_ARCHIVE_ID:", RESTORE_FINTECH_ARCHIVE_ID)
print("RESTORE_STRATLAKE_SESSION_ID:", RESTORE_STRATLAKE_SESSION_ID)
print("RESTORE_STRATLAKE_ARCHIVE_ID:", RESTORE_STRATLAKE_ARCHIVE_ID)
print("Restore Fintech session path exists:", RESTORE_FINTECH_DRIVE_SESSION_ROOT.exists())
print("Restore Fintech backup pack path exists:", RESTORE_FINTECH_BACKUP_PACK_DIR.exists())
print("Restore StratLake session path exists:", RESTORE_STRATLAKE_DRIVE_SESSION_ROOT.exists())
print("Restore StratLake archive pack path exists:", RESTORE_STRATLAKE_ARCHIVE_PACK_DIR.exists())

## Fintech curated-data archive command previews

Notebook 04 does not need to create or restore the Fintech curated-data archive by default. That belongs to the Fintech storage notebooks.

This cell prints the session-derived command templates so the StratLake feature-series setup remains aligned with the latest Fintech archive deployment.

Use these previews when a new Colab runtime needs to hydrate `MARKETLAKE_ROOT` from a previous `{FINTECH_SESSION_ID}` archive before generating StratLake features.

In [ ]:
fintech_pack_preview = f'''
fintech-backup-data pack \
  --root {FINTECH_ROOT_STR} \
  --dataset-root {MARKETLAKE_ROOT_STR} \
  --archive-id {FINTECH_ARCHIVE_ID} \
  --drive-root {FINTECH_DRIVE_BACKUP_ROOT_STR} \
  --copy-policy overwrite_allowed \
  --validate-after-copy \
  --inspect-after-copy
'''.strip()

fintech_restore_preview = f'''
fintech-backup-data restore \
  --root {FINTECH_ROOT_STR} \
  --archive-id {RESTORE_FINTECH_ARCHIVE_ID} \
  --drive-root {RESTORE_FINTECH_DRIVE_BACKUP_ROOT_STR} \
  --target-root {MARKETLAKE_ROOT_STR} \
  --copy-policy overwrite_allowed \
  --validate-after-copy \
  --inspect-after-copy
'''.strip()

print("Fintech archive pack preview:")
print(fintech_pack_preview)
print("\nFintech archive restore preview:")
print(fintech_restore_preview)

## Optional: verify or restore Fintech curated data before StratLake feature generation

Run this optional cell only when `MARKETLAKE_ROOT` is empty and you already have a previous Fintech archive pack on Drive.

The default is intentionally a preview. Uncomment the command only after setting `RESTORE_FINTECH_SESSION_ID` / `RESTORE_FINTECH_ARCHIVE_ID` to the archive you want to use.

In [ ]:
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT)
print("MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())

if MARKETLAKE_ROOT.exists():
    sample_files = list(MARKETLAKE_ROOT.rglob("*"))[:10]
    print("Sample curated-data paths:")
    for path in sample_files:
        print(" -", path)
else:
    print("No curated-data root found yet. Restore from a Fintech archive before running StratLake feature generation.")

# Example restore command, intentionally commented:
#
# !fintech-backup-data restore \
#   --root {FINTECH_ROOT_STR} \
#   --archive-id {RESTORE_FINTECH_ARCHIVE_ID} \
#   --drive-root {RESTORE_FINTECH_DRIVE_BACKUP_ROOT_STR} \
#   --target-root {MARKETLAKE_ROOT_STR} \
#   --copy-policy overwrite_allowed \
#   --validate-after-copy \
#   --inspect-after-copy

## Shared readiness check

This check confirms that both notebook session IDs, session-scoped Drive folders, archive identifiers, and the explicit `MARKETLAKE_ROOT` connection are available.

`MARKETLAKE_ROOT` may be empty in a brand-new runtime until you run the earlier Fintech extraction notebook or restore a Fintech archive pack.

In [ ]:
print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("FINTECH_ARCHIVE_ID:", FINTECH_ARCHIVE_ID)
print("STRATLAKE_ARCHIVE_ID:", STRATLAKE_ARCHIVE_ID)

print("\nLocal workspace checks:")
print("FINTECH_ROOT exists:", FINTECH_ROOT.exists())
print("STRATLAKE_ROOT exists:", STRATLAKE_ROOT.exists())
print("MARKETLAKE_ROOT exists:", MARKETLAKE_ROOT.exists())

print("\nDrive session checks:")
print("FINTECH_DRIVE_SESSION_ROOT exists:", FINTECH_DRIVE_SESSION_ROOT.exists())
print("STRATLAKE_DRIVE_SESSION_ROOT exists:", STRATLAKE_DRIVE_SESSION_ROOT.exists())
print("FINTECH_DRIVE_BACKUP_ROOT exists:", FINTECH_DRIVE_BACKUP_ROOT.exists())
print("STRATLAKE_DRIVE_ARCHIVE_ROOT exists:", STRATLAKE_DRIVE_ARCHIVE_ROOT.exists())

print("\nArchive pack path checks:")
print("FINTECH_BACKUP_PACK_DIR:", FINTECH_BACKUP_PACK_DIR)
print("FINTECH_BACKUP_PACK_DIR exists:", FINTECH_BACKUP_PACK_DIR.exists())
print("STRATLAKE_ARCHIVE_PACK_DIR:", STRATLAKE_ARCHIVE_PACK_DIR)
print("STRATLAKE_ARCHIVE_PACK_DIR exists:", STRATLAKE_ARCHIVE_PACK_DIR.exists())

print("\nExplicit data handoff:")
print("StratLake marketlake_root ->", MARKETLAKE_ROOT)

## Next notebook

Continue to **Notebook 05 — StratLake Q1 Feature Data Generation**.

Notebook 05 should reuse:

```python
FINTECH_SESSION_ID
STRATLAKE_SESSION_ID
MARKETLAKE_ROOT
STRATLAKE_ROOT
STRATLAKE_DRIVE_SESSION_ROOT
```

Notebook 05 focuses on generating feature data from Q1 market data using StratLake while keeping the upstream Fintech curated-data session identity explicit.